The goal by the end of this is for me to verify the transaction myself for a confirmed transaction. I have chosen this transaction: <a class="transaction-id" href="https://mempool.space/tx/29a3efd3ef04f9153d47a990bd7b048a4b2d213daaa5fb8ed670fb85f13bdbcf" target="_blank" rel="noopener noreferrer">29a3efd3ef04f9153d47a990bd7b048a4b2d213daaa5fb8ed670fb85f13bdbcf</a>

This transaction was confirmed in block 153509 on 16 November 2011. It has 11 inputs and two outputs: 50,000 BTC and 500,000 BTC. From these 11 inputs, I will find the highest value input, then check the output related to that, and verify the spend manually via Python. I'm not using Bitcoin libraries such as python-bitcoinlib for interacting with Bitcoin core as the objective is to DIY and learn.



## My plan (without AI, before coding)

1. We are given the transaction hash, and the block height.
2. From the block height, we get the block hash from our local pruned node itself.
3. Our pruned node doesn't have access to transactions in the old block. So, we call getpeerinfo, which lists peers already connected to our node. We are looking for a peer that has NETWORK as the service name, because it means it can supply historical blocks.
4. Once we find a peer, we request the block from one of those peers.
5. Once we receive the block, we get the chosen transaction from this block.
6. We read the transaction’s raw bytes from the block stored locally. The command displayed those bytes as hex text.
7. We decode the transaction’s hex text into bytes.
8. From this transaction, we also get all the <del>inputs it accessed to make this transaction</del> transaction inputs.
9. For this exercise, we will choose one transaction input which has the most funds.
10. So, first, for each of this transactions inputs, <del>we need to get the previous transactions which approved spending these inputs</del> we look at the previous\_output (outpoint), which contains the previous transaction id and the output index. (We will use the explorer only to find the block hash of each of the tx input. The transaction bytes would still come from the peer.)
11. We fetch any missing blocks from peers for these previous transactions.
12. <del>For each input, we use its previous transaction ID and output index to find the exact output.</del> We decode the blocks which contains the previous transactions, and for each of the transactions, we look into the outputs of each one, and get the value (output amount) in Satoshis. We read that output’s amount. We compare the 11 amounts and choose the largest referenced output.
13. For the largest output, we get the <del>spending conditions</del> pk\_script which defines the spending conditions.
14. For the largest output, we find the input in the main transaction that referenced it.
15. From that input, we get the signature and public key it provided to spend the previous output.
16. So, now we have the spending conditions for the previous output, and the signature and public key from the newer input.
17. We check that the public key satisfies the relevant condition in the previous output.
18. We construct the exact message hash for the chosen input’s signature.
19. We verify the signature using that public key and message hash.



## Coding (with AI help)

<a href="https://github.com/arjunrajkumar/exploring-bitcoin/tree/main/projects/transaction-signature-verification" target="_blank" rel="noopener noreferrer">Github</a>



## What I learnt

- The blockchain is a database, but it's not like a normal DB. Instead, it's more like a snapshot to verify that things existed in a state on a previous date. For example, there is no simple query to find which block a transaction exists on! There is no direct lookup that a node can perform. So, Bitcoin Core creates two databases, a UTXO DB (outputs that have not been spent), and an optional transaction index DB. Because I'm running a pruned node, I don't have the transaction index DB.
- Because of the pruned node, I have to request any transaction, blocks, etc. from peers who are running a full archival node. But I still can't request a transaction from a peer without a block hash.
- Everything in the blockchain is stored in bytes. The bitcoin-cli and API convert it into hex text while displaying it for me.

  <div class="byte-example">
  <div class="byte-example-row">
  <code>00000000000005c6aca89f188819553e63fe1d5deeea9722b43d1cd0974c7d58</code>
  <span class="byte-example-label">Block hash in hex</span>
  </div>
  <div class="byte-example-row">
  <code>01 00 00 00 f6 05 25 43 53 a7 6d 40 6d 62 8a 86 45 f7 5a 4b f6 48 64 61 c8 1e 76 b7 60 02 00 00 00 00 00 00 4f c5 cd ab 3d 9d 02 e4 bc 62 0a b7 97 80 9c 88 95 d7 6e 5b 9a 54 8c a9 8d f2 86 00 ec 8c 5f 98 23 d5 6c 4e e3 6f 09 1a 23 d7 3f e4</code>
  <span class="byte-example-label">Block header in hex</span>
  </div>
  </div>
- In a transaction, an input spends a previous output. That previous output held the funds, and it has some spending conditions, which the spender has to pass to spend its funds. The input doesn't contain the amount it spent. It only has a reference to the previous output, which has the amount it spent.
- When I downloaded Bitcoin Core, it also provided me with an API called JSON-RPC. Basically, I can send a JSON request to my local Bitcoin Core, and it runs it and will return a response.
- Each block has a block header (80 bytes), which consists of the version (4 bytes), previous block header hash (32 bytes), merkle root hash (32 bytes), time (4 bytes), nBits (4 bytes), and nonce (4 bytes).

  <div class="byte-example">
  <div class="byte-example-row">
  <code>01 00 00 00</code>
  <span class="byte-example-label">version (4 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>f6 05 25 43 53 a7 6d 40 6d 62 8a 86 45 f7 5a 4b f6 48 64 61 c8 1e 76 b7 60 02 00 00 00 00 00 00</code>
  <span class="byte-example-label">previous block header hash (32 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>4f c5 cd ab 3d 9d 02 e4 bc 62 0a b7 97 80 9c 88 95 d7 6e 5b 9a 54 8c a9 8d f2 86 00 ec 8c 5f 98</code>
  <span class="byte-example-label">merkle root hash (32 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>23 d5 6c 4e</code>
  <span class="byte-example-label">time (4 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>e3 6f 09 1a</code>
  <span class="byte-example-label">nBits (4 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>23 d7 3f e4</code>
  <span class="byte-example-label">nonce (4 bytes)</span>
  </div>
  </div>
- After the 80-byte block header, the next bytes represent the transaction count, which tells us how many transactions are included in the block. But the transaction count uses CompactSize which is not fixed. Basically, if the first byte falls between 00-'fc', then it means read only the 81st byte. If the first byte is 'fd', then read the next 2 bytes (82nd and 83rd), for 'fe' read next 4 bytes, for 'ff' read next 8 bytes. Or if it's easier to think in integers.. 0-252 is first.. then 253, 254 and 255 are the markers.
- I’m not using decoderawtransaction from Bitcoin core for this problem.. So, looking up a previous transaction in an older block isn't very straightforward. For eg, each outpoint in a transaction input has the previous transaction id. BUT, this transaction id is actually a hash of all the data of that previous transaction. It doesn’t exist inside the previous transaction. It's there in the blocks merkle root which allows us to verify that the transaction exists in the block. But as I want to find selected transactions to read the outputs from it, I have to hash each one of the blocks transactions, create the transaction hash first, and if chosen, get the transaction outputs. For each transaction I want, the block already has the exact bytes I need to hash. But I need to find the start and position for each transaction from the block.
- To find the start, end for each transaction.. I need to increment the position by the correct field size. Some of the transaction fields are or fixed bytes.. e.g. version, lock\_time, the previous\_output etc.. But many of them are of variable size of type compactSize. For e.g. the number of transactions in a block, the number of input transactions in a transaction, the public\_key script. So, the program needs to update the position by finding the compactSize for each of them. Doing all of this will give us the start and end position for each transaction.. Which we can then use to find the exact bytes for each transaction and then hash it to get the id.



In [4]:
#| echo: true
#| output: true

print("Hello world!")

Hello world!
